# Board deck KPIs — as it was handed over

A colleague started this notebook for the board deck and has moved to another team. Four KPIs are drafted; two are
not started. **What each KPI means is in `BRIEF.md`: read it before anything else.** The brief is the definition; the
code below is only your colleague's attempt at it.

The cells down to **Your work starts here** are your colleague's. They run without an error. Leave them as they are:
they are the "before" that your diagnosis note refers to. Your repairs go in new cells below.

In [ ]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

## The headline numbers (supplied; these are right)

Every KPI below is a breakdown or an average of one of these. They come from one table each, with no join.

In [ ]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM 'data/raw/orders.csv')                        AS orders,
           (SELECT COUNT(DISTINCT order_id) FROM 'data/raw/order_payments.csv') AS paid_orders,
           (SELECT SUM(payment_value) FROM 'data/raw/order_payments.csv')      AS revenue_brl,
           (SELECT COUNT(*) FROM 'data/raw/order_items.csv')                   AS items,
           (SELECT SUM(price) FROM 'data/raw/order_items.csv')                 AS item_sales_brl
""").df()

## The exchange rates (supplied; this table is right)

The ECB's rates, cached from the Frankfurter API in `data/raw/api/frankfurter_eur.json` (Lab 4's file). `rates` has one
row per ECB business day per currency, and `eur_per_unit` is the euros one unit of that currency buys.

In [ ]:
raw = json.load(open("data/raw/api/frankfurter_eur.json"))
assert raw["base"] == "EUR", raw["base"]     # each value is units of the currency for raw["amount"] euros

rows = []
for day, values in raw["rates"].items():
    for currency, units_per_euro in values.items():
        rows.append((day, currency, raw["amount"] / units_per_euro))

con.execute("CREATE OR REPLACE TABLE rates (date DATE, currency VARCHAR, eur_per_unit DOUBLE)")
con.executemany("INSERT INTO rates VALUES (?, ?, ?)", rows)
print(con.sql("SELECT COUNT(*), COUNT(DISTINCT date), MIN(date), MAX(date) FROM rates").fetchone())

## The KPIs as handed over

### KPI 1. Monthly revenue, in reais and in euros

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE draft_revenue_month AS
    WITH order_revenue AS (
        SELECT order_id, SUM(payment_value) AS revenue_brl
        FROM 'data/raw/order_payments.csv'
        GROUP BY order_id
    )
    SELECT date_trunc('month', o.order_purchase_timestamp) AS month,
           COUNT(*)                                        AS orders,
           SUM(r.revenue_brl)                              AS revenue_brl,
           SUM(r.revenue_brl * x.eur_per_unit)             AS revenue_eur
    FROM 'data/raw/orders.csv' o
    JOIN order_revenue r USING (order_id)
    JOIN rates x ON x.date = CAST(o.order_purchase_timestamp AS DATE) AND x.currency = 'BRL'
    GROUP BY month
    ORDER BY month
""")
brl, eur, n = con.sql("SELECT SUM(revenue_brl), SUM(revenue_eur), SUM(orders) FROM draft_revenue_month").fetchone()
print(f"KPI 1 total: {brl:,.2f} BRL = {eur:,.2f} EUR over {n:,} orders")
con.sql("SELECT * FROM draft_revenue_month").df()

### KPI 2. Average order value by customer state

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE draft_aov_state AS
    SELECT c.customer_state,
           COUNT(*)     AS orders,
           AVG(i.price) AS avg_order_value
    FROM 'data/raw/orders.csv' o
    JOIN 'data/raw/customers.csv' c USING (customer_id)
    JOIN 'data/raw/order_items.csv' i USING (order_id)
    GROUP BY c.customer_state
    ORDER BY orders DESC
""")
overall = con.sql("""
    SELECT AVG(i.price) FROM 'data/raw/orders.csv' o JOIN 'data/raw/order_items.csv' i USING (order_id)
""").fetchone()[0]
n = con.sql("SELECT SUM(orders) FROM draft_aov_state").fetchone()[0]
print(f"KPI 2: average order value, all states, {overall:,.2f} BRL over {n:,} orders")
con.sql("SELECT * FROM draft_aov_state").df()

### KPI 3. Average review score by customer state

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE draft_review_state AS
    SELECT c.customer_state,
           COUNT(*)            AS reviews,
           AVG(r.review_score) AS avg_review_score
    FROM 'data/raw/order_reviews.csv' r
    JOIN 'data/raw/orders.csv' o USING (order_id)
    JOIN 'data/raw/customers.csv' c USING (customer_id)
    GROUP BY c.customer_state
    ORDER BY reviews DESC
""")
n, overall = con.sql("SELECT COUNT(*), AVG(review_score) FROM 'data/raw/order_reviews.csv'").fetchone()
print(f"KPI 3: {n:,} reviews, average review score, all states, {overall:.3f}")
con.sql("SELECT * FROM draft_review_state").df()

### KPI 4. Item sales by product category

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE draft_category AS
    SELECT t.product_category_name_english AS category,
           COUNT(*)                        AS items,
           SUM(i.price)                    AS sales_brl
    FROM 'data/raw/order_items.csv' i
    JOIN 'data/raw/products.csv' p USING (product_id)
    JOIN 'data/raw/product_category_name_translation.csv' t USING (product_category_name)
    GROUP BY category
    ORDER BY sales_brl DESC
""")
items, sales, cats = con.sql("SELECT SUM(items), SUM(sales_brl), COUNT(*) FROM draft_category").fetchone()
print(f"KPI 4 total: {items:,} items, {sales:,.2f} BRL of item sales, in {cats} categories")
con.sql("SELECT * FROM draft_category").df()

### KPI 5. Delivery time by customer state — not started

### KPI 6. Revenue by seller state, in euros — not started

Both are defined in `BRIEF.md`. They are yours: section C below.

---

# Your work starts here

The order of work is in the README. **Section A comes before section B**: it measures your colleague's drafts as they
are, and its cells stay as they are after you repair, so they remain the evidence. Every cell below has an ID in its
first line (`A1`, `B1`, `C1`, `D`, `E`); `DIAGNOSIS.md` cites cells and joins by ID instead of pasting them again.

## A. The evidence, before any repair

For each drafted KPI: **the three counts for every join it makes** (rows in the left table; rows in the result;
distinct values of the left table's key in the result), and the key test (`COUNT(*)` against `COUNT(DISTINCT …)`) on
any table whose grain you are not sure of. Add as many cells as you need; one per KPI is a good start. Write each
join's three counts **once**, in the Joins section of `DIAGNOSIS.md`, with an ID (`J1`, `J2`, …): the notes cite it.

In [ ]:
# A1. KPI 1: the three counts for each of its joins. Which orders did the join to rates lose, and on which days?

In [ ]:
# A2. KPI 2: the three counts for its join to order_items. What is one row of order_items?

In [ ]:
# A3. KPI 3: what is one row of order_reviews? The key test, and the three counts for its join to orders.

In [ ]:
# A4. KPI 4: the three counts for each of its joins. How many items did the join to the translation table lose?

## B. The repairs, each a query that follows the brief

Each repair makes a table with the name and columns given, so that sections D and E can check it. Add the three counts
of every join you write to the Joins section of `DIAGNOSIS.md`, one line each, with its ID; the notes and your commit
messages cite the ID.

### B1. KPI 1 → `kpi_revenue_month` (`month`, `orders`, `revenue_brl`, `revenue_eur`)

The brief's conversion rule, one row per month, every paid order exactly once.

In [ ]:
# B1. CREATE OR REPLACE TABLE kpi_revenue_month AS ...

### B2. KPI 2 → `kpi_aov_state` (`customer_state`, `orders`, `revenue_brl`, `avg_order_value`)

In [ ]:
# B2. CREATE OR REPLACE TABLE kpi_aov_state AS ...

### B3. KPI 3 → `kpi_review_state` (`customer_state`, `reviews`, `avg_review_score`)

In [ ]:
# B3. CREATE OR REPLACE TABLE kpi_review_state AS ...

### B4. KPI 4, in two queries

First, **the anti-join**: which categories have no English name, and how many items and reais does each carry?
Then **`kpi_category`** (`category`, `items`, `sales_brl`), in which every item appears exactly once, labelled as the
brief says.

In [ ]:
# B4, first query: the anti-join.

In [ ]:
# B4, second query: CREATE OR REPLACE TABLE kpi_category AS ...

## C. The two new KPIs, from the brief

Their joins go in the Joins section of `DIAGNOSIS.md` too, each with its ID.

### C1. KPI 5 → `kpi_delivery_state` (`customer_state`, `orders`, `avg_delivery_days`)

In [ ]:
# C1. CREATE OR REPLACE TABLE kpi_delivery_state AS ...

### C2. KPI 6 → `kpi_seller_state_eur` (`seller_state`, `items`, `sales_brl`, `sales_eur`)

In [ ]:
# C2. CREATE OR REPLACE TABLE kpi_seller_state_eur AS ...

## D. The reconciliation table

One row per KPI (KPI 1 gets two: reais and euros), plus the EUR/BRL plausibility bound. Each row names **the identity
that KPI allows** — a sum; a ratio, with its numerator and its denominator; or a weighted mean — and puts the KPI
table's number beside a number computed **independently**, from the source table and by different code. Print it as
a table. **Tolerances:** money agrees to the cent — `abs(a - b) < 0.005`, or both rounded to two decimals — never
`==` on sums of money, which are floats; whole-number counts agree exactly; means to six decimals.

In [ ]:
# D. The reconciliation table.

## E. The assertions

One cell that stops with a message if any identity in section D fails — with the tolerances above, never `==` on
money — and two more: **no order without a rate**
(no order and no item has a NULL euro value) and **no order counted twice** (every order-grain table has one row per
order, and the monthly table counts each paid order once). The message carries the number, so a failure says what
broke.

In [ ]:
# E. The assertions.

## Stretch (optional, not graded)

The brief counts **one review, one vote**. A different, defensible rule is **one order, one vote**: average each
order's reviews first, then average the orders. Compute KPI 3 that way. Which states' averages move by more than 0.05?
Would the board's reading change?

In [ ]:
# Stretch.